<a href="https://colab.research.google.com/github/nuhuynhh/AAI2026/blob/main/ML_Basics_Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Data source: Kaggle - Telco Customer Churn Dataset
# https://www.kaggle.com/datasets/blastchar/telco-customer-churn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Checked churnn distribution
df["Churn"].value_counts()

,count
Churn,
No,5174
Yes,1869


In [9]:
df_model = df[
    [
         "SeniorCitizen",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "Contract",
        "InternetService",
        "Churn"
    ]
].copy()

# COnvert churn to 1/0 (1 = churned, 0 = not churned)
df_model["Churn"] = df_model["Churn"].map({"Yes": 1, "No": 0})

df_model["TotalCharges"] = pd.to_numeric(df_model["TotalCharges"], errors="coerce")
df_model = df_model.dropna()

In [17]:
# features and target
X = df_model.drop("Churn", axis=1)
y = df_model["Churn"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# preprocessing: scale numerical features and one-hot encode categorical features
numeric_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = ["Contract", "InternetService"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(sparse_output=False), categorical_features)
    ]
)

In [24]:
# Pipeline with preprocessing and regression model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42, max_iter=500))
])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# train the model
model.fit(X_train, y_train)

# Predict churn probability for a new customer
new_customer = pd.DataFrame({
    "SeniorCitizen": [0],          # 0 = not senior, 1 = senior
    "tenure": [12],
    "MonthlyCharges": [85],
    "TotalCharges": [900],
    "Contract": ["Month-to-month"],
    "InternetService": ["Fiber optic"]
})
churn_probability = model.predict_proba(new_customer)[0][1]

# Classify based on threshold
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0

print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")

Churn Probability for new customer: 0.58
Churn Prediction (1 = churn, 0 = no churn): 1


In [25]:
# Model Coeficcients:
cat_feature_names = model.named_steps["preprocessor"] \
    .named_transformers_["cat"] \
    .get_feature_names_out(categorical_features)

feature_names = list(cat_feature_names) + numeric_features
coefficients = model.named_steps["classifier"].coef_[0]

print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")


Model Coefficients:
Contract_Month-to-month: 0.14
Contract_One year: -1.36
Contract_Two year: -0.06
InternetService_DSL: 0.65
InternetService_Fiber optic: 0.88
InternetService_No: -0.05
SeniorCitizen: -0.83
tenure: -0.00
MonthlyCharges: 1.01
TotalCharges: -1.01
